In [15]:
import pandas as pd
import numpy as np

from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

from pprint import pprint

In [16]:
df = pd.DataFrame({
    'gender': ['Female', 'Male', 'Male', 'Female', 'Unknown'],
    'age':    [22      , None,      33    , 44      , 28      ],
    'city':   ['Hamburg', 'Warsaw', 'Beijin', 'Hamburg', 'Oslo']
})
df_test = pd.DataFrame({
    'gender': ['Female'],
    'age': [np.nan],
    'city': ['Hamburg']
})

In [17]:
df

,gender,age,city
0,Female,22.0,Hamburg
1,Male,NaN,Warsaw
2,Male,33.0,Beijin
3,Female,44.0,Hamburg
4,Unknown,28.0,Oslo


In [18]:
df_test

,gender,age,city
0,Female,NaN,Hamburg


In [19]:
from sklearn.pipeline import make_pipeline, make_union, Pipeline, FeatureUnion

# Feature union

- Concat results of multiple transormer objects. 
- make_union - convenience function
- combines several transformer objects into a new transformer that combines their output. 
- The transformers are applied in parallel,


In [20]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
union = make_union(
    SimpleImputer(strategy='constant', fill_value=1), 
    StandardScaler(),
    
    verbose=True
)

In [21]:
union.fit_transform(df.age.to_frame().values)

[FeatureUnion] . (step 1 of 2) Processing simpleimputer, total=   0.0s
[FeatureUnion]  (step 2 of 2) Processing standardscaler, total=   0.0s


array([[22.        , -1.20759819],
       [ 1.        ,         nan],
       [33.        ,  0.15482028],
       [44.        ,  1.51723875],
       [28.        , -0.46446084]])

In [22]:
union.transform(df_test.age.to_frame().values)

array([[ 1., nan]])

# Pipeline
- Sequentially apply a list of transforms and a final estimator
- The final estimator only needs to implement fit
- The transformers in the pipeline can be cached using memory argument.
- The purpose of the pipeline is to assemble several steps that can be cross-validated together while setting different parameters

In [23]:
from sklearn.base import TransformerMixin, BaseEstimator
class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, name):
        self.name=name

    def fit(self, df, *args, **kwargs):
        return self
        
    
    def transform(self, df, *args, **kwargs):
        return df[self.name].to_frame().values
    

In [26]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer, ColumnTransformer, make_column_selector

gender_f = make_pipeline(
    FeatureSelector('gender'),
    OneHotEncoder(), 
    verbose=True,
    memory=cache_dir
)

In [27]:
gender_f.fit_transform(df)

[Pipeline] ... (step 1 of 2) Processing featureselector, total=   0.0s
[Pipeline] ..... (step 2 of 2) Processing onehotencoder, total=   0.0s


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 3)>

In [28]:
gender_f.transform(df_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1 stored elements and shape (1, 3)>

In [29]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing  import StandardScaler, Normalizer
age_f = make_pipeline(
    FeatureSelector('age'),
    SimpleImputer(strategy='median'), 
    StandardScaler(), 
    
    verbose=True,
    memory=cache_dir)

In [30]:
age_f.fit_transform(df)

[Pipeline] ... (step 1 of 3) Processing featureselector, total=   0.0s
[Pipeline] ..... (step 2 of 3) Processing simpleimputer, total=   0.0s
[Pipeline] .... (step 3 of 3) Processing standardscaler, total=   0.0s


array([[-1.31237504],
       [-0.13814474],
       [ 0.20721711],
       [ 1.72680926],
       [-0.48350659]])

In [31]:
features = make_union(gender_f, age_f, verbose=True)

In [32]:
features.fit_transform(df)

[Pipeline] ..... (step 2 of 2) Processing onehotencoder, total=   0.0s
[FeatureUnion] .... (step 1 of 2) Processing pipeline-1, total=   0.0s
[Pipeline] ..... (step 2 of 3) Processing simpleimputer, total=   0.0s
[Pipeline] .... (step 3 of 3) Processing standardscaler, total=   0.0s
[FeatureUnion] .... (step 2 of 2) Processing pipeline-2, total=   0.0s


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10 stored elements and shape (5, 4)>

In [33]:
from sklearn.utils import estimator_html_repr
from IPython.core.display import HTML
HTML(estimator_html_repr(features))

,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('pipeline-1', ...), ('pipeline-2', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",True
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,name,'gender'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an u